# 지능형 고객 응대 Chatbot 구현

## 환경 설정

In [1]:
# 필요한 함수 / 클래스 임포트
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

In [2]:
# .env file -> API Key 가져오기
env_file_path='.env'
output = load_dotenv(env_file_path)
print(output)

True


## 데이터 불러오기

In [ ]:
# 내용 정리

"""
1. TextLoader 클래스
   1) .txt 파일에 있는 데이터 로드 및 자료형 변환 클래스
   2) 기능1: 외부 파일(.txt 파일)로부터 텍스트 데이터 추출
   3) 기능2: 텍스트 데이터를 LangChain의 기본 자료형 Document 자료형으로 변환
2. Document 자료형
   1) 텍스트 데이터에 대한 LangChain의 기본 자료형
   2) 메타 데이터(metadata: 파일 경로 포함) + 텍스트 내용(page_content)으로 구성
3. 전용 함수: load() 함수 -> TextLoader 클래스 호출 시 지정된 파일을 encoding 방식으로 읽고 Document 자료형으로 변환      
"""

# 데이터 로드 및 변환 도구 생성
loader = TextLoader(file_path="C:/Users/playdata2/work/manual.txt", encoding="utf-8")

# load() 함수 호출, Document 자료형 생성 -> 결과물: 리스트 [Document], .txt 파일 1개당 Document 자료형 1개 생성
documents = loader.load()
print(f'데이터 로드 및 변환의 결과물: \n{documents}') 



데이터 로드 및 변환의 결과물: 
[Document(metadata={'source': 'C:/Users/playdata2/work/manual.txt'}, page_content="질문: 코드크래프터스 아카데미에 어떻게 가입하나요?\n답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.\n\n질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\n답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.\n\n질문: 강의를 다 들으면 수료증을 받을 수 있나요?\n답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실 수 있습니다.\n\n질문: 어떤 결제 수단을 사용할 수 있나요?\n답변: 저희는 학습자님의 편의를 위해 신용카드, 실시간 계좌이체, 그리고 가상계좌 입금 등 다양한 결제 방법을 지원하고 있습니다. 원하시는 결제 수단을 선택하여 진행해 주세요.\n\n질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.\n답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제가 지속될 경우, 즉시 기술 지원팀에 문의해 주세요.\n\n질문: 강의 자료나 소스 코드는 어디서 다운로드받나요?\n답변: 각 강의의 첫 번째 챕터인 '강의 소개' 섹션과 각 챕터의 하단에서 관련 PDF 교안 및 전체 소스 코드를 압축 파일 형태로 다운로드하실 수 있습니다.\n\n질문: 수강 기간에 제한이 있나요?\n답변: 아니요, 저희 코드크래프터스 아카데미의 모

## 텍스트 분할

In [12]:
# 적절한 글자수(chunk_size)를 찾기 위한 계산 단계

## 파이썬 문법을 이용, .txt 파일의 데이터를 문자열로 변환
file_path='manual.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    manual_text = f.read()

print(manual_text)

print('-'*80)

## 데이터 상태 확인 -> 각 질문-답변 쌍 -> 문단 단위('\n\n')로 구분

qa_pairs = []

for pair in manual_text.split("\n\n"):
    qa_pairs.append(pair)  

print(qa_pairs)

print('-'*80)

## 각 QA 쌍 -> 글자수 계산
character_length=[]

for pair in qa_pairs:
    character_length.append(len(pair))

print(f'각 질문-답변 쌍의 글자수: \n{character_length}')

print('-'*80)

##  QA 쌍 중 글자 수가 가장 긴 경우의 글자수 추출
max_char_length = max(character_length)
print(f'가장 긴 QA 단락의 글자 수: {max_char_length}자')

질문: 코드크래프터스 아카데미에 어떻게 가입하나요?
답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.

질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.

질문: 강의를 다 들으면 수료증을 받을 수 있나요?
답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실 수 있습니다.

질문: 어떤 결제 수단을 사용할 수 있나요?
답변: 저희는 학습자님의 편의를 위해 신용카드, 실시간 계좌이체, 그리고 가상계좌 입금 등 다양한 결제 방법을 지원하고 있습니다. 원하시는 결제 수단을 선택하여 진행해 주세요.

질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.
답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제가 지속될 경우, 즉시 기술 지원팀에 문의해 주세요.

질문: 강의 자료나 소스 코드는 어디서 다운로드받나요?
답변: 각 강의의 첫 번째 챕터인 '강의 소개' 섹션과 각 챕터의 하단에서 관련 PDF 교안 및 전체 소스 코드를 압축 파일 형태로 다운로드하실 수 있습니다.

질문: 수강 기간에 제한이 있나요?
답변: 아니요, 저희 코드크래프터스 아카데미의 모든 강의는 한 번 구매하시면 평생 소장이 가능합니다. 시간과 장소에 구애받지 않고 원하실 때 언제든지 복습하실 수 있습니다.

질문: 코딩 실습 중에 모르는 것이 생기면 어떻게 하죠?
답변: 각 강의 영상 하단에 있는

In [10]:
for text in qa_pairs:
    print(text)
    print('-'*80)

질문: 코드크래프터스 아카데미에 어떻게 가입하나요?
답변: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.
--------------------------------------------------------------------------------
질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.
--------------------------------------------------------------------------------
질문: 강의를 다 들으면 수료증을 받을 수 있나요?
답변: 물론입니다. 각 강의의 모든 챕터를 100% 수강 완료하시면, 즉시 온라인으로 발급되는 공식 수료증을 받으실 수 있습니다. 수료증은 PDF 파일로 다운로드하여 소장하거나 제출용으로 활용하실 수 있습니다.
--------------------------------------------------------------------------------
질문: 어떤 결제 수단을 사용할 수 있나요?
답변: 저희는 학습자님의 편의를 위해 신용카드, 실시간 계좌이체, 그리고 가상계좌 입금 등 다양한 결제 방법을 지원하고 있습니다. 원하시는 결제 수단을 선택하여 진행해 주세요.
--------------------------------------------------------------------------------
질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.
답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제

In [13]:
# RecursiveCharacterTextSplitter 클래스 내용 정리
"""
1. 임베딩 모델이 한 번에 처리할 수 있는 입력 길이 제한(8,192 토큰)을 고려하여 긴 문서를 
   작은 조각(chunk)로 분할하거나 짧은 문서 조각을 서로 병합하여 사용자 질문과 유사도 계산
   시 정확도를 높일 수 있는 적절한 길이로 만든다.
2. RecursiveCharacterTextSplitter
   1) 데이터 분할 도구
   2) 분할 순서
      - 기본 설정: \n\n 단락(문단), \n문장,공백(" "), 글자 단위("")  
      - 먼저 가장 큰 단위인 \n\n(단락)을 기준으로 텍스트를 쪼갠다
      - 쪼개진 텍스트 조각이 여전히 사용자가 설정한 기준 길이(chunk_size)보다 크다면, 
        그 다음 구분자인 \n(줄바꿈/문장)으로 다시 쪼갠다
      - 설정한 크기 이하가 될 때까지 이 과정을 공백( ), 글자 단위("") 순으로 반복한다.
      - 기준 길이(chunk_size)보다 짧으면 병합
   3) Document 자료형의 텍스트 데이터를 분할
3. 매개 변수 정리
   1) chunk_size: 각 청크의 최대 문자 수
   2) chunk_overlap: 인접한 청크 간의 중복 문자 수 -> QA, 법률 -> overlap=0 / 일반 소설 -> chunk_size 10% 
4.split_documents() 함수
   1) RecursiveCharacterTextSplitter 전용 함수
   2) 입력된 Document 자료형의 텍스트 데이터를 설정 기준(chunk_size, chunk_overlap)에 따라 분할
"""

# 데이터 분할 도구 생성
text_splitter = RecursiveCharacterTextSplitter(    
    chunk_size=160,
    chunk_overlap=0
)